In [1]:
import numpy as np
import pandas as pd
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent   # /mnt/primary/Main
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from loader.load import load_aligned_plans_and_runs

from uncertainty_prediction.src import *
from uncertainty_prediction.new_config import *

from uncertainty_prediction.baselines.deterministic.tnn.model import TNNRuntimeRegressor
from uncertainty_prediction.baselines.deterministic.tlstm.data import get_train_test_datasets_from_trino, tlstm_collate_fn_identity
from uncertainty_prediction.baselines.deterministic.tlstm.util import unnormalize_labels

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from torch.utils.data import DataLoader
import torch.nn.functional as F

set_seed(42)

In [2]:

"""
Load query plan and runs
"""
# any set of plans is fine here (... assuming that plans are identical across runs)
queries_dir = "/mnt/lakehouse-raw-results/tpcds/lakehouse-a/20260429-135222Z/queries"

plans_by_query, runs_by_query, common = load_aligned_plans_and_runs(
    queries_dir=queries_dir,
    run_ids=RUN_IDS,
    collection=COLLECTION_NAME,
    schema=SCHEMA_NAME,
    instance=LAKEHOUSE_INSTANCE_NAME,
    metric=METRIC,
    xcol=XCOL,
    ycol=YCOL,
    parsed_results_root=PARSED_RESULTS_ROOT,
    canon_fn=canon_qid,
    min_runs=1,
    min_points_per_run=2,
    require_cols=(XCOL, YCOL),
)

In [3]:

"""
Split into training and testing sets
"""

train_qids, test_qids = split_query_ids(
    common,
    seed=SEED,
    test_frac=TEST_FRAC,
)
print("n_train:", len(train_qids), "n_test:", len(test_qids))

runs_train, runs_test = make_train_test_run_split(
    runs_by_query,
    train_qids=train_qids,
    test_qids=test_qids,
)


n_train: 351 n_test: 87


In [4]:

"""
Generate training and testing sets 
"""

tlstm_data = get_train_test_datasets_from_trino(
    plans_by_query=plans_by_query,
    runs_by_query=runs_by_query,
    train_qids=train_qids,
    test_qids=test_qids,
    xcol=XCOL,              # e.g. "t_rel_s"
    runtime_mode="mean",    # mean runtime across repeated runs
    batch_size=64,          # number of queries merged into one TLSTM batch
    condition_max_num=8,    # number of tokens per condition stream per node
    condition_op_dim=16,    # size of each token in condition1/condition2
    extra_dim=16,           # size of extra_infos vector
    sample_dim=32,          # size of sample branch vector
)

train_dataset = tlstm_data["train_dataset"]
test_dataset = tlstm_data["test_dataset"]

train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    collate_fn=tlstm_collate_fn_identity,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=tlstm_collate_fn_identity,
)

In [5]:

"""
Build model
"""

device = "cuda" if torch.cuda.is_available() else "cpu"

model = TNNRuntimeRegressor(
    operator_dim=tlstm_data["operator_dim"],
    extra_dim=tlstm_data["extra_dim"],
    condition_op_dim=tlstm_data["condition_op_dim"],
    sample_dim=tlstm_data["sample_dim"],
    hidden_dim=128,
    hid_dim=256,
    head_dim=256,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

In [6]:

"""
Train model
"""

num_epochs = 30
best_val_loss = float("inf")
best_state = None

for epoch in range(num_epochs):
    model.train()
    train_losses = []

    for batch in train_loader:
        operators = batch["operators"].to(device)
        extra_infos = batch["extra_infos"].to(device)
        condition1s = batch["condition1s"].to(device)
        condition2s = batch["condition2s"].to(device)
        samples = batch["samples"].to(device)
        condition_masks = batch["condition_masks"].to(device).unsqueeze(2)  # [L, N, 1]
        mapping = batch["mapping"].to(device)
        targets = batch["target_runtime"].to(device).reshape(-1, 1)

        optimizer.zero_grad()

        preds = model(
            operators,
            extra_infos,
            condition1s,
            condition2s,
            samples,
            condition_masks,
            mapping,
        )

        loss = F.mse_loss(preds, targets)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

    model.eval()
    val_losses = []

    with torch.no_grad():
        for batch in test_loader:
            operators = batch["operators"].to(device)
            extra_infos = batch["extra_infos"].to(device)
            condition1s = batch["condition1s"].to(device)
            condition2s = batch["condition2s"].to(device)
            samples = batch["samples"].to(device)
            condition_masks = batch["condition_masks"].to(device).unsqueeze(2)
            mapping = batch["mapping"].to(device)
            targets = batch["target_runtime"].to(device).reshape(-1, 1)

            preds = model(
                operators,
                extra_infos,
                condition1s,
                condition2s,
                samples,
                condition_masks,
                mapping,
            )

            loss = F.mse_loss(preds, targets)
            val_losses.append(loss.item())

    mean_train = float(np.mean(train_losses))
    mean_val = float(np.mean(val_losses))
    print(f"Epoch {epoch+1:03d} | train_loss={mean_train:.6f} | val_loss={mean_val:.6f}")

    if mean_val < best_val_loss:
        best_val_loss = mean_val
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

if best_state is not None:
    model.load_state_dict(best_state)

/opt/miniconda3/lib/python3.10/site-packages/torch/nn/modules/linear.py:125: UserWarning: Could not parse CUBLAS_WORKSPACE_CONFIG, using default workspace size of 8519680 bytes. (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:143.)
  return F.linear(input, self.weight, self.bias)
/opt/miniconda3/lib/python3.10/site-packages/torch/nn/modules/linear.py:125: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA >= 10.2. To enable deterministic behavior in this case, you must set an environment variable before running your PyTorch application: CUBLAS_WORKSPACE_CONFIG=:4096:8 or CUBLAS_WORKSPACE_CONFIG=:16:8. For more information, go to https://docs.nvidia.com/cuda/cublas/index.html#results-reproducibility (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:233.)
  return F.line

Epoch 001 | train_loss=0.068881 | val_loss=0.099491
Epoch 002 | train_loss=0.057146 | val_loss=0.087719
Epoch 003 | train_loss=0.056972 | val_loss=0.075566
Epoch 004 | train_loss=0.056572 | val_loss=0.065370
Epoch 005 | train_loss=0.057217 | val_loss=0.057972
Epoch 006 | train_loss=0.057498 | val_loss=0.054452
Epoch 007 | train_loss=0.055146 | val_loss=0.056972
Epoch 008 | train_loss=0.054716 | val_loss=0.066084
Epoch 009 | train_loss=0.058285 | val_loss=0.071099
Epoch 010 | train_loss=0.055164 | val_loss=0.078592
Epoch 011 | train_loss=0.053836 | val_loss=0.087004
Epoch 012 | train_loss=0.054930 | val_loss=0.079478
Epoch 013 | train_loss=0.053650 | val_loss=0.083540
Epoch 014 | train_loss=0.053969 | val_loss=0.079420
Epoch 015 | train_loss=0.052853 | val_loss=0.060194
Epoch 016 | train_loss=0.052485 | val_loss=0.056445
Epoch 017 | train_loss=0.053758 | val_loss=0.053788
Epoch 018 | train_loss=0.053453 | val_loss=0.052371
Epoch 019 | train_loss=0.052879 | val_loss=0.052327
Epoch 020 | 

In [7]:

"""
Evaluate
"""

def qerror_numpy(y_pred, y_true, eps=1e-12):
    y_pred = np.asarray(y_pred, dtype=float).reshape(-1)
    y_true = np.asarray(y_true, dtype=float).reshape(-1)
    return np.maximum((y_pred + eps) / (y_true + eps), (y_true + eps) / (y_pred + eps))

min_val, max_val = tlstm_data["label_stats"]

all_preds_norm = []
all_targets_norm = []
all_qids = []

model.eval()
with torch.no_grad():
    for batch in test_loader:
        preds = model(
            batch["operators"].to(device),
            batch["extra_infos"].to(device),
            batch["condition1s"].to(device),
            batch["condition2s"].to(device),
            batch["samples"].to(device),
            batch["condition_masks"].to(device).unsqueeze(2),
            batch["mapping"].to(device),
        )

        all_preds_norm.append(preds.cpu().numpy().reshape(-1))
        all_targets_norm.append(batch["target_runtime"].cpu().numpy().reshape(-1))
        all_qids.extend(batch["qids"])

preds_norm = np.concatenate(all_preds_norm)
targets_norm = np.concatenate(all_targets_norm)

preds = unnormalize_labels(preds_norm, min_val, max_val)
targets = unnormalize_labels(targets_norm, min_val, max_val)

qerr = qerror_numpy(preds, targets)

metrics = {
    "mae": float(mean_absolute_error(targets, preds)),
    "rmse": float(np.sqrt(mean_squared_error(targets, preds))),
    "r2": float(r2_score(targets, preds)),
    "median_q_error": float(np.median(qerr)),
    "p95_q_error": float(np.percentile(qerr, 95)),
    "mean_q_error": float(np.mean(qerr)),
    "gmean_q_error": float(np.exp(np.mean(np.log(qerr + 1e-12)))),
}
print(metrics)

{'mae': 67.9959487915039, 'rmse': 272.7060717375028, 'r2': -0.06125473976135254, 'median_q_error': 4.021815146721222, 'p95_q_error': 45.32904127309938, 'mean_q_error': 17.611753117952276, 'gmean_q_error': 4.506981436367355}


In [8]:
from scipy.stats import spearmanr
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def print_metrics_for_excel(metrics):
    print("\t".join(str(v) for v in metrics.values()))

def print_metric_headers_for_excel(metrics):
    print("\t".join(metrics.keys()))
    
def evaluate_runtime_metrics(preds_unnorm, labels_unnorm, eps=1e-12):
    preds_unnorm = np.asarray(preds_unnorm, dtype=float).reshape(-1)
    labels_unnorm = np.asarray(labels_unnorm, dtype=float).reshape(-1)

    if preds_unnorm.shape[0] != labels_unnorm.shape[0]:
        raise ValueError("preds_unnorm and labels_unnorm must have the same length")

    qerr = qerror_numpy(preds_unnorm, labels_unnorm)

    spearman = spearmanr(labels_unnorm, preds_unnorm).statistic
    if spearman is None or np.isnan(spearman):
        spearman = 0.0

    metrics = {
        "mae": float(mean_absolute_error(labels_unnorm, preds_unnorm)),
        "rmse": float(np.sqrt(mean_squared_error(labels_unnorm, preds_unnorm))),
        "spearman": float(spearman),
        "mean_q_error": float(np.mean(qerr)),
        "median_q_error": float(np.median(qerr)),
        "p90_q_error": float(np.percentile(qerr, 90)),
        "p95_q_error": float(np.percentile(qerr, 95)),
    }
    return metrics

test_metrics = evaluate_runtime_metrics(preds, targets)

print_metrics_for_excel(test_metrics)



67.99594761927922	272.7060681498962	0.2250461544876244	17.611753117952276	4.021815146721222	20.834541145112166	45.32904127309938
